In [ ]:
from ultralytics import YOLO
from pathlib import Path

BASE_DIR = Path.cwd()
PRETRAINED_DIR = BASE_DIR.parent / "runs" / "detect" / "pcb_defect_detector_v13" / "weights" / "best.pt"
RUNS_DIR = BASE_DIR.parent / "runs" / "valid"
DATA_CONFIG = BASE_DIR.parent / "datasets" / "pcb-defects-final" / "data.yaml"

model = YOLO(str(PRETRAINED_DIR)) 

metrics = model.val(data=str(DATA_CONFIG),  project=str(RUNS_DIR),) 

print(f"Средняя точность (mAP@50-95): {metrics.box.map}")
print(f"mAP@50: {metrics.box.map50}")
print(f"mAP@75: {metrics.box.map75}")

In [ ]:
from ultralytics import YOLO
from pathlib import Path

BASE_DIR = Path.cwd()
PRETRAINED_DIR = BASE_DIR.parent / "runs" / "detect" / "pcb_defect_detector_v13" / "weights" / "best.pt"
RUNS_DIR = BASE_DIR.parent / "runs" / "predict"
VIDEO_CONFIG = BASE_DIR.parent / "datasets" / "test_video_pcb_defects_2.mp4"

model = YOLO(str(PRETRAINED_DIR))

# model.export(format="engine", half=True, device=0)

results = model.predict(
    source=str(VIDEO_CONFIG),
    project=str(RUNS_DIR),
    imgsz=640,
    conf=0.5,
    save=True
)

In [ ]:
import torch
import cv2
import numpy as np
import os
import segmentation_models_pytorch as smp

# 1. Настройки
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_path = '../runs/detect/u_net_resnet34_weights/best.pth'
test_img_dir = '../datasets/pcb-defects-final/val/images'
save_res_dir = '../runs/valid/u_net_resnet34'
os.makedirs(save_res_dir, exist_ok=True)

# 2. Загрузка модели
model = smp.Unet(
    encoder_name="resnet34", 
    encoder_weights=None,
    in_channels=3, 
    classes=1
).to(device)

model.load_state_dict(torch.load(model_path))
model.eval()

# 3. Функция для обработки
def predict(image_path):
    # Загрузка и предобработка
    img_orig = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    
    # Ресайз под вход модели
    img_input = cv2.resize(img_rgb, (640, 640))
    img_input = img_input.transpose(2, 0, 1) / 255.0
    img_input = torch.from_numpy(img_input).float().unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(img_input)
        mask_pred = torch.sigmoid(output).squeeze().cpu().numpy()
        
    # Бинаризация
    mask_binary = (mask_pred > 0.8).astype(np.uint8) * 255
    
    # Ресайз маски обратно к оригинальному размеру фото
    mask_binary = cv2.resize(mask_binary, (w, h), interpolation=cv2.INTER_NEAREST)
    
    # Создаем наложение (overlay) - дефекты будут красными
    overlay = img_orig.copy()
    overlay[mask_binary > 0] = [0, 0, 255] # BGR (красный)
    
    # Соединяем оригинал и результат для сравнения
    combined = np.hstack((img_orig, overlay))
    return combined

# 4. Запуск теста
img_list = os.listdir(test_img_dir)[:10] # Протестируем первые 10 картинок
print(f"Тестирование на {len(img_list)} изображениях...")

for name in img_list:
    full_path = os.path.join(test_img_dir, name)
    result_img = predict(full_path)
    cv2.imwrite(os.path.join(save_res_dir, f"res_{name}"), result_img)

print(f"Результаты сохранены в папку: {save_res_dir}")